### Bronze Level Data

* This notebook is to treat the raw data from the *towed ADCP* and prepare ***Bronze*** and ***Silver*** level data matrices.

---
`MoeinDst, version 2.4, 09.07.24, Faro, Portugal`

In [ ]:
import numpy as np
import scipy.io
import pandas as pd
import os, sys

In [ ]:
sTA1 = scipy.io.loadmat('/home/moein/UAlg_CIMA/Data/STA1.mat')
sTA2 = scipy.io.loadmat('/home/moein/UAlg_CIMA/Data/STA2.mat')
# sTA3 = scipy.io.loadmat('/home/moein/data_July2024/Data/STA3.mat') # problematic, needs special treatment
sTA4 = scipy.io.loadmat('/home/moein/UAlg_CIMA/Data/STA4.mat')

In [ ]:
def datenum(year, month, day, hour, minute, second):
    
    year = 2000 + year.flatten() # Since year is recorded as 24
    dates = pd.to_datetime({'year': year, 'month': month.flatten(), 'day': day.flatten(), 'hour': hour.flatten(), 'minute': minute.flatten(), 'second': second.flatten()})
    return dates

In [ ]:
time1 = datenum(sTA1['SerYear'], sTA1['SerMon'], sTA1['SerDay'], sTA1['SerHour'], sTA1['SerMin'], sTA1['SerSec'])
time2 = datenum(sTA2['SerYear'], sTA2['SerMon'], sTA2['SerDay'], sTA2['SerHour'], sTA2['SerMin'], sTA2['SerSec'])
time4 = datenum(sTA4['SerYear'], sTA4['SerMon'], sTA4['SerDay'], sTA4['SerHour'], sTA4['SerMin'], sTA4['SerSec'])

Time = np.concatenate((time1, time2, time4))

In [ ]:
Lon1 = (sTA1['AnLLonDeg'] + sTA1['AnFLonDeg']) / 2
Lon2 = (sTA2['AnLLonDeg'] + sTA2['AnFLonDeg']) / 2
Lon4 = (sTA4['AnLLonDeg'] + sTA4['AnFLonDeg']) / 2
Lon = np.concatenate((Lon1, Lon2, Lon4))

Lat1 = (sTA1['AnLLatDeg'] + sTA1['AnFLatDeg']) / 2
Lat2 = (sTA2['AnLLatDeg'] + sTA2['AnFLatDeg']) / 2
Lat4 = (sTA4['AnLLatDeg'] + sTA4['AnFLatDeg']) / 2
Lat = np.concatenate((Lat1, Lat2, Lat4))

In [ ]:
def quality_control(data, error_field, quality_field):
    # Extract the fields for error and quality
    error_data = np.array(data[error_field]).astype(float)
    quality_data = np.array(data[quality_field]).astype(float)

    # Create a mask based on the conditions
    mask = (quality_data > 70) | (error_data < 10)
    
    # Apply the mask to the error data
    filtered_error_data = np.where(mask, error_data, np.nan)
    
    return filtered_error_data

In [ ]:
### --- Eastward velocity --- ###
u1 = quality_control(sTA1, 'SerEmmpersec', 'SerPG4')
u1 = np.where(u1==-32768, np.nan,u1)
u2 = quality_control(sTA2, 'SerEmmpersec', 'SerPG4')
u2 = np.where(u2==-32768, np.nan,u2)
u4 = quality_control(sTA4, 'SerEmmpersec', 'SerPG4')
u4 = np.where(u4==-32768, np.nan,u4)

u1 = np.hstack((u1, np.full((u1.shape[0], 4), np.nan)))
u2 = np.hstack((u2, np.full((u2.shape[0], 4), np.nan)))
U = np.concatenate((u1, u2, u4)) / 1000

### --- Northward velocity --- ###
v1 = quality_control(sTA1, 'SerNmmpersec', 'SerPG4')
v1 = np.where(v1==-32768, np.nan,v1)
v2 = quality_control(sTA2, 'SerNmmpersec', 'SerPG4')
v2 = np.where(v2==-32768, np.nan,v2)
v4 = quality_control(sTA4, 'SerNmmpersec', 'SerPG4')
v4 = np.where(v4==-32768, np.nan,v4)

v1 = np.hstack((v1, np.full((v1.shape[0], 4), np.nan)))
v2 = np.hstack((v2, np.full((v2.shape[0], 4), np.nan)))
V = np.concatenate((v1, v2, v4)) / 1000

In [ ]:
### --- Depth --- ###
Depth1 = (sTA1['AnBTDepthcmB1'] + sTA1['AnBTDepthcmB2'] + sTA1['AnBTDepthcmB3'] + sTA1['AnBTDepthcmB4']) / 400
Depth2 = (sTA2['AnBTDepthcmB1'] + sTA2['AnBTDepthcmB2'] + sTA2['AnBTDepthcmB3'] + sTA2['AnBTDepthcmB4']) / 400
Depth4 = (sTA4['AnBTDepthcmB1'] + sTA4['AnBTDepthcmB2'] + sTA4['AnBTDepthcmB3'] + sTA4['AnBTDepthcmB4']) / 400

Depth = np.concatenate((Depth1, Depth2, Depth4))

In [ ]:
### --- Temperature --- ###
## -- converting units
Temp = np.concatenate((sTA1['AnT100thDeg'] / 100, sTA2['AnT100thDeg'] / 100, sTA4['AnT100thDeg'] / 100))
## -- filtering erroneous temperature records
Temp[Temp > 26] = np.nan

In [ ]:
### Bin Depth
BinDepth = []

for depth in Depth:
    r = depth / 34
    r1 = np.arange(r, depth +r- 0.0001, r)
    BinDepth.append(r1)

BinDepthArray = np.array(BinDepth)

In [ ]:
### --- Bakcscatter records from instrument --- ###

BackScatt1 = sTA1['SerEAAcnt']
BackScatt2 = sTA2['SerEAAcnt']
BackScatt4 = sTA4['SerEAAcnt']

BackScatt1 = np.hstack((BackScatt1, np.full((BackScatt1.shape[0], 4), np.nan)))
BackScatt2 = np.hstack((BackScatt2, np.full((BackScatt2.shape[0], 4), np.nan)))

BackScatter = np.concatenate((BackScatt1, BackScatt2, BackScatt4))

In [ ]:
### --- Bottom-track velocities --- ###

u_bt1 = sTA1['AnBTEmmpersec']
u_bt2 = sTA2['AnBTEmmpersec']
u_bt4 = sTA4['AnBTEmmpersec']
u_bt = np.concatenate((u_bt1, u_bt2, u_bt4))/1000

v_bt1 = sTA1['AnBTNmmpersec']
v_bt2 = sTA2['AnBTNmmpersec']
v_bt4 = sTA4['AnBTNmmpersec']
v_bt = np.concatenate((v_bt1, v_bt2, v_bt4))/1000

In [ ]:
data = []

# Iterate through each time step
for i in range(Time.shape[0]):
    data.append(
        [Time[i], Lon[i], Lat[i], u_bt[i], v_bt[i], BackScatter[i],
         U[i, :], V[i, :], BinDepthArray[i,:],
        Temp[i]]
    )

columns_list = ['Date-Time', 'Longitude', 'Latitude',
                'u_bt', 'v_bt','BackScatter',
                'U-Eastward [m/s]', 'V-Northward [m/s]',
                'BinDepth[m]', 'Temperature']
# Creating a DataFrame out of that
df = pd.DataFrame(data, columns=columns_list)
desired_order = [
    'Date-Time', 'Longitude', 'Latitude', 'BinDepth[m]',
    'U-Eastward [m/s]','V-Northward [m/s]', 'u_bt', 'v_bt',
    'BackScatter', 'Temperature'
]
df_bronze = df[desired_order]

In [ ]:
makeplot = False
if makeplot == True:
    import matplotlib.pyplot as plt

In [ ]:
if makeplot:
    fig, ax = plt.subplots()
    ax3.scatter(df_bronze['u_bt'],df_bronze['v_bt'])
    ax3.set_xlabel('u_bt')
    ax3.set_ylabel('v_bt')
    ax3.set_title('Scatter Plot of bottom track velocities')

In [ ]:
u_lev1 = []
v_lev1 = []
for r in range(df_bronze['U-Eastward [m/s]'].shape[0]):
    u_lev1.append(df_bronze['U-Eastward [m/s]'][r][0])
    
for r in range(df_bronze['V-Northward [m/s]'].shape[0]):
    v_lev1.append(df_bronze['V-Northward [m/s]'][r][0])

if makeplot:
    fig, ax = plt.subplots()
    ax4.scatter(u_lev1, v_lev1)
    ax.grid(ls='--', alpha=0.3)

### Silver Matrix

The silver step estimates a heading/rotation correction (`alpha`) and scale correction (`beta`) for each transect by comparing bottom-track velocities with navigation velocities. Most of the times -- unless there are marked compass interferences and biases -- values of $\alpha$ varies between $\pm 0.2$ radians, while those of $\beta$ fall between $\pm 0.03$.


The same correction is then applied to every measured water-velocity profile (`U-Eastward [m/s]`, `V-Northward [m/s]`) inside that transect segment.

In [ ]:
transect_indices = np.array([
    # Transect 1
    [76, 231], [525, 698], [999, 1174], [1474, 1645], [2009, 2218], [2529, 2769],
    [3143, 3337], [3669, 3903], [4186, 4331], [4595, 4831], [5149, 5313], [5638, 5759],
    [6092, 6232], [6496, 6622], [6873, 6995], [7300, 7414], [7661, 7778], [8036, 8163],
    [8396, 8490], [8711, 8833], [9126, 9248],
    # Transect 2
    [231, 380], [698, 860], [1174, 1343], [1645, 1878], [2218, 2398], [2769, 2962],
    [3337, 3512], [3903, 4025], [4331, 4443], [4831, 5003], [5313, 5474], [5759, 5944],
    [6232, 6362], [6622, 6766], [6995, 7132], [7414, 7528], [7778, 7905], [8163, 8297],
    [8490, 8599], [8833, 8986], [9248, 9347],
    # Transect 3
    [380, 532], [860, 1005], [1343, 1474], [1878, 2019], [2398, 2544], [2962, 3148],
    [3512, 3678], [4025, 4192], [4443, 4608], [5003, 5151], [5474, 5640], [5944, 6098],
    [6362, 6506], [6766, 6877], [7132, 7300], [7528, 7663], [7905, 8037], [8297, 8407],
    [8599, 8713], [8986, 9130], [9347, 9443]
])

In [ ]:
# Navigation velocities [m/s]
u_sh1 = sTA1['AnNVEmmpersec']
u_sh2 = sTA2['AnNVEmmpersec']
u_sh4 = sTA4['AnNVEmmpersec']
u_sh = np.concatenate((u_sh1, u_sh2, u_sh4)).astype(float).squeeze() / 1000

v_sh1 = sTA1['AnNVNmmpersec']
v_sh2 = sTA2['AnNVNmmpersec']
v_sh4 = sTA4['AnNVNmmpersec']
v_sh = np.concatenate((v_sh1, v_sh2, v_sh4)).astype(float).squeeze() / 1000

# Bottom-track velocities already loaded in df_bronze [m/s]
u_bt = np.asarray(df_bronze['u_bt'].values, dtype=float).squeeze()
v_bt = np.asarray(df_bronze['v_bt'].values, dtype=float).squeeze()

In [ ]:
def compute_alpha_beta(transect_indices, u_bt, v_bt, u_sh, v_sh):
    """
    Compute heading/rotation correction alpha [rad] and scale correction beta [-]
    for each transect interval.

    alpha is computed with arctan2(mean(cross), mean(dot)) so that the quadrant is preserved.
    beta is the fractional speed-scale difference between navigation and bottom-track speeds.
    """
    alpha = np.full(transect_indices.shape[0], np.nan, dtype=float)
    beta = np.full(transect_indices.shape[0], np.nan, dtype=float)

    for i, (start_idx, end_idx) in enumerate(transect_indices):
        idx = np.arange(start_idx, end_idx + 1)

        cross = u_bt[idx] * v_sh[idx] - v_bt[idx] * u_sh[idx]
        dot = u_bt[idx] * u_sh[idx] + v_bt[idx] * v_sh[idx]

        alpha[i] = np.arctan2(np.nanmean(cross), np.nanmean(dot))

        nav_speed2 = u_sh[idx]**2 + v_sh[idx]**2
        bt_speed2 = u_bt[idx]**2 + v_bt[idx]**2
        denom = np.nanmean(bt_speed2)
        beta[i] = np.sqrt(np.nanmean(nav_speed2) / denom) - 1 if denom > 0 else np.nan

    return alpha, beta


alpha, beta = compute_alpha_beta(transect_indices, u_bt, v_bt, u_sh, v_sh)

alpha_transect1 = alpha[0:21]
alpha_transect2 = alpha[21:42]
alpha_transect3 = alpha[42:63]

beta_transect1 = beta[0:21]
beta_transect2 = beta[21:42]
beta_transect3 = beta[42:63]

silver_corrections = pd.DataFrame({
    'transect_group': np.repeat([1, 2, 3], 21),
    'transect_repeat': np.tile(np.arange(1, 22), 3),
    'start_idx': transect_indices[:, 0],
    'end_idx': transect_indices[:, 1],
    'alpha_rad': alpha,
    'alpha_deg': np.rad2deg(alpha),
    'beta_rad': beta,
    'beta_deg': np.rad2deg(beta),
})


In [ ]:
silver_corrections.head()

In [ ]:
### Quick Diagnostic Check -- in terms of the range for alpha and beta:
silver_corrections[['alpha_rad', 'alpha_deg', 'beta_rad', 'beta_deg']].describe()

In [ ]:
def rotate_and_scale_profile(u_profile, v_profile, alpha, beta):
    """
    Apply the silver-level correction to one vertical velocity profile.
    
    The function was created by ChatGPT AI-agent, that used the manual workflow
    and turned it to a cleaner function that serves the same purpose.

    Parameters
    ----------
    u_profile, v_profile : array-like
        Eastward and northward water velocity components for all bins at one timestamp.
    alpha : float
        Heading/rotation correction in radians.
    beta : float
        Fractional scale correction.

    Returns
    -------
    u_corr, v_corr : np.ndarray
        Corrected eastward and northward velocity profiles.
    """
    u = np.asarray(u_profile, dtype=float)
    v = np.asarray(v_profile, dtype=float)

    scale = 1 + beta
    u_corr = scale * (u * np.cos(alpha) - v * np.sin(alpha))
    v_corr = scale * (u * np.sin(alpha) + v * np.cos(alpha))

    return u_corr, v_corr


def apply_silver_correction(df_bronze, transect_indices, alpha, beta):
    """
    Creates a silver-level dataframe by applying alpha/beta correction to every row
    that falls inside one of the transect intervals.
    """
    df_silver = df_bronze.copy()

    n_rows = len(df_silver)
    u_silver = [np.full_like(np.asarray(x, dtype=float), np.nan) for x in df_silver['U-Eastward [m/s]']]
    v_silver = [np.full_like(np.asarray(x, dtype=float), np.nan) for x in df_silver['V-Northward [m/s]']]
    alpha_by_row = np.full(n_rows, np.nan, dtype=float)
    beta_by_row = np.full(n_rows, np.nan, dtype=float)
    transect_group_by_row = np.full(n_rows, np.nan, dtype=float)
    transect_repeat_by_row = np.full(n_rows, np.nan, dtype=float)

    for i, (start_idx, end_idx) in enumerate(transect_indices):
        # Clip intervals to the dataframe length in case the notebook is run on a shorter subset.
        start_idx = max(int(start_idx), 0)
        end_idx = min(int(end_idx), n_rows - 1)

        group = (i // 21) + 1
        repeat = (i % 21) + 1

        for row_idx in range(start_idx, end_idx + 1):
            u_corr, v_corr = rotate_and_scale_profile(
                df_silver.at[row_idx, 'U-Eastward [m/s]'],
                df_silver.at[row_idx, 'V-Northward [m/s]'],
                alpha[i],
                beta[i]
            )
            u_silver[row_idx] = u_corr
            v_silver[row_idx] = v_corr
            alpha_by_row[row_idx] = alpha[i]
            beta_by_row[row_idx] = beta[i]
            transect_group_by_row[row_idx] = group
            transect_repeat_by_row[row_idx] = repeat

    df_silver['transect_group'] = transect_group_by_row
    df_silver['transect_repeat'] = transect_repeat_by_row
    df_silver['alpha_rad'] = alpha_by_row
    df_silver['beta_rad'] = beta_by_row
    df_silver['U-Eastward silver [m/s]'] = u_silver
    df_silver['V-Northward silver [m/s]'] = v_silver

    return df_silver

In [ ]:
df_silver = apply_silver_correction(df_bronze, transect_indices, alpha, beta)
df_silver

* The missing values or NaNs for several first Date-Times are associated with the deployment and adjustment of the instrument at the very beginning.

                        <-------------------- End of Document -------------------->